In [1]:
from futpythontrader import *

# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

colecao_partidas1 = TinyDB('./base_flashscore_2019.json')
colecao_partidas2 = TinyDB('./base_flashscore_2020.json')
colecao_partidas3 = TinyDB('./base_flashscore_2021.json')
# colecao_partidas4 = TinyDB('./base_flashscore_2022.json')
# colecao_partidas5 = TinyDB('./base_flashscore_2023.json')
# colecao_partidas6 = TinyDB('./base_flashscore_2024.json')
colecao_partidas7 = TinyDB('./base_flashscore_2021_2022.json')
colecao_partidas8 = TinyDB('./base_flashscore_2022_2023.json')
# colecao_partidas9 = TinyDB('./base_flashscore_2023_2024.json')

df1 = pd.DataFrame(colecao_partidas1.all())
df2 = pd.DataFrame(colecao_partidas2.all())
df3 = pd.DataFrame(colecao_partidas3.all())
# df4 = pd.DataFrame(colecao_partidas4.all())
# df5 = pd.DataFrame(colecao_partidas5.all())
# df6 = pd.DataFrame(colecao_partidas6.all())
df7 = pd.DataFrame(colecao_partidas7.all())
df8 = pd.DataFrame(colecao_partidas8.all())
# df9 = pd.DataFrame(colecao_partidas9.all())

# df = pd.concat([df1,df2,df3,df4,df5,df6,df7,df8,df9])
df = pd.concat([df1,df2,df3,df7,df8])
df.drop_duplicates(subset=['Id'], inplace=True)

df.columns.to_list()

# df[['Dia','Mes','Ano']] = df['Date'].str.split('/',expand=True)
# df = df.drop(['Date'], axis=1)
# df['Date'] = df['Ano']+'-'+df['Mes']+'-'+ df['Dia']
# df = df.drop(columns=['Dia','Mes','Ano'], axis=1)

# df = df[['Id','Date','Time','League','Home','Away','Round','Season',
#          'Odd_H','Odd_D','Odd_A','Odd_1X','Odd_12','Odd_X2',
#          'Odd_Over25','Odd_Under25','Odd_BTTS_Yes','Odd_BTTS_No',
#          'Goals_H_HT','Goals_A_HT','Goals_H_FT','Goals_A_FT','Goals_Minutes_Home','Goals_Minutes_Away']]

# df.columns = ['Id','Date','Time','League','Home','Away','Round','Season',
#               'Odd_H','Odd_D','Odd_A','Odd_LayAway','Odd_LayDraw','Odd_LayHome',
#               'Odd_Over25','Odd_Under25','Odd_BTTS_Yes','Odd_BTTS_No',
#               'Goals_H_HT','Goals_A_HT','Goals_H_FT','Goals_A_FT','Goals_Minutes_Home','Goals_Minutes_Away']

# df.fillna(0, inplace=True)

# df = df.sort_values(['Date','Time'])

# df = drop_reset_index(df)

# df.to_csv('C:\\GitHub\\YouTube\\Luke\\Bases_de_Dados\\Base_de_Dados_FutPythonPunter.csv',index=False)

['Id',
 'Date',
 'Time',
 'League',
 'Home',
 'Away',
 'Round',
 'Season',
 'Goals_H_HT',
 'Goals_A_HT',
 'Goals_H_FT',
 'Goals_A_FT',
 'Odd_H',
 'Odd_D',
 'Odd_A',
 'Odd_Over25',
 'Odd_Under25',
 'Odd_BTTS_Yes',
 'Odd_BTTS_No',
 'Odd_1X',
 'Odd_12',
 'Odd_X2',
 'Goals_Minutes_Home',
 'Goals_Minutes_Away']

In [3]:
df['Result'] = 'D'
df.loc[(df['Goals_H_FT'] > df['Goals_A_FT']), 'Result'] = 'H'
df.loc[(df['Goals_H_FT'] < df['Goals_A_FT']), 'Result'] = 'A'

df['LayAway'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] >= row['Goals_A_FT']) else 0, axis=1)
df['LayDraw'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] != row['Goals_A_FT']) else 0, axis=1)
df['LayHome'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] <= row['Goals_A_FT']) else 0, axis=1)

df['Lay_0x1'] = df.apply(lambda row: 0 if ((row['Goals_H_FT'] == 0) & (row['Goals_A_FT'] == 1)) else 1, axis=1)
df['Lay_0x2'] = df.apply(lambda row: 0 if ((row['Goals_H_FT'] == 0) & (row['Goals_A_FT'] == 2)) else 1, axis=1)
df['Lay_1x0'] = df.apply(lambda row: 0 if ((row['Goals_H_FT'] == 1) & (row['Goals_A_FT'] == 0)) else 1, axis=1)
df['Lay_2x0'] = df.apply(lambda row: 0 if ((row['Goals_H_FT'] == 2) & (row['Goals_A_FT'] == 0)) else 1, axis=1)

df['Profit_LayAway'] = df.apply(lambda row: (row['Odd_LayAway'] - 1) if (row['LayAway'] == 1) else -1, axis=1)
df['Profit_LayDraw'] = df.apply(lambda row: (row['Odd_LayDraw'] - 1) if (row['LayDraw'] == 1) else -1, axis=1)
df['Profit_LayHome'] = df.apply(lambda row: (row['Odd_LayHome'] - 1) if (row['LayHome'] == 1) else -1, axis=1)

df['VAR1'] = np.sqrt((df['Odd_H'] - df['Odd_A'])**2 + (df['Odd_H'] - df['Odd_A'])**2)
df['VAR2'] = np.degrees(np.arctan((df['Odd_A'] - df['Odd_H']) / 2))
df['VAR3'] = np.degrees(np.arctan((df['Odd_D'] - df['Odd_A']) / 2))

df.to_excel('Base_Lay.xlsx',index=False)


In [ ]:
base = df.copy()

In [ ]:
base['League'].unique()

In [ ]:
liga = 'ENGLAND - LEAGUE ONE'

df = base[base.League == liga]
df = drop_reset_index(df)
df = df[['League','Date','Home','Away','Odd_LayAway','Odd_LayDraw','Odd_LayHome','Result','Profit_LayAway','Profit_LayDraw','Profit_LayHome']]
df

In [ ]:
no_of_days = 0

matchDates = df.Date.unique()

if no_of_days > 0:
    matchDates = (matchDates[-no_of_days:])

df2 = pd.DataFrame()

rowsIndex = []
rowsDate = []
rowsH = []
rowsD = []
rowsA = []

count = 0
for mDate in matchDates:
    count += 1
    rowsDate.append(mDate)
    rowsH.append(df.loc[df['Date'] == mDate]['Profit_LayHome'].sum())
    rowsD.append(df.loc[df['Date'] == mDate]['Profit_LayDraw'].sum())
    rowsA.append(df.loc[df['Date'] == mDate]['Profit_LayAway'].sum())

df2['Date'] = rowsDate
df2['H'] = rowsH
df2['D'] = rowsD
df2['A'] = rowsA

df2 = df2.tail(101)
df2 = df2.reset_index(drop=True)
df2['Id'] = df2.reset_index()['index'].rename('index_copy')
df2['Id'] = df2['Id'] + 1
df2 = df2[['Id','Date','H','D','A']]

df2['Hacu'] = df2.H.cumsum()
df2['Dacu'] = df2.D.cumsum()
df2['Aacu'] = df2.A.cumsum()

df2['Hacu'].loc[0] = np.nan
df2['Dacu'].loc[0] = np.nan
df2['Aacu'].loc[0] = np.nan

def weighted_mean_H(s):
    d = df2.loc[s.index, 'Hacu']
    w = df2.loc[s.index, 'Id']
    return (d * w).sum() / w.sum()

def weighted_mean_H_C(s):
    d = df2.loc[s.index, 'waHC']
    w = df2.loc[s.index, 'Id']
    return (d * w).sum() / w.sum()

def weighted_mean_D(s):
    d = df2.loc[s.index, 'Dacu']
    w = df2.loc[s.index, 'Id']
    return (d * w).sum() / w.sum()

def weighted_mean_D_C(s):
    d = df2.loc[s.index, 'waDC']
    w = df2.loc[s.index, 'Id']
    return (d * w).sum() / w.sum()

def weighted_mean_A(s):
    d = df2.loc[s.index, 'Aacu']
    w = df2.loc[s.index, 'Id']
    return (d * w).sum() / w.sum()

def weighted_mean_A_C(s):
    d = df2.loc[s.index, 'waAC']
    w = df2.loc[s.index, 'Id']
    return (d * w).sum() / w.sum()

df2['waH16'] = df2.rolling(16)['Hacu'].apply(weighted_mean_H, raw=False)
df2['waH8'] = df2.rolling(8)['Hacu'].apply(weighted_mean_H, raw=False)
df2['waHC'] = 2*df2.waH8-df2.waH16
df2['waH4'] = df2.rolling(4)['waHC'].apply(weighted_mean_H_C, raw=False)

df2['waD16'] = df2.rolling(16)['Dacu'].apply(weighted_mean_D, raw=False)
df2['waD8'] = df2.rolling(8)['Dacu'].apply(weighted_mean_D, raw=False)
df2['waDC'] = 2*df2.waD8-df2.waD16
df2['waD4'] = df2.rolling(4)['waDC'].apply(weighted_mean_D_C, raw=False)

df2['waA16'] = df2.rolling(16)['Aacu'].apply(weighted_mean_A, raw=False)
df2['waA8'] = df2.rolling(8)['Aacu'].apply(weighted_mean_A, raw=False)
df2['waAC'] = 2*df2.waA8-df2.waA16
df2['waA4'] = df2.rolling(4)['waAC'].apply(weighted_mean_A_C, raw=False)

df2['Hhull'] = df2['waH4']
df2['Dhull'] = df2['waD4']
df2['Ahull'] = df2['waA4']

df2 = df2[['Id','Date','H','D','A','Hacu','Dacu','Aacu','Hhull','Dhull','Ahull']]
df2

In [ ]:
# df2.Hacu.plot()
# df2.Hhull.plot()

# df2.Dacu.plot()
# df2.Dhull.plot()

df2.Aacu.plot()
df2.Ahull.plot()

In [ ]:
def TR_Match_Odds(base, jogos_do_dia, lista):

    ligas = jogos_do_dia.sort_values(['League'])
    ligas = ligas['League'].unique()

    for i in ligas:
        liga = i

        df = base[base.League == liga]
        # print(liga)

        df["Goals_H"] = df["Goals_H"].astype('int64')
        df["Goals_A"] = df["Goals_A"].astype('int64')
        df['Odd_H'] = pd.to_numeric(df['Odd_H'],errors = 'coerce')
        df['Odd_D'] = pd.to_numeric(df['Odd_D'],errors = 'coerce')
        df['Odd_A'] = pd.to_numeric(df['Odd_A'],errors = 'coerce')

        total_count = len(df.index)
        #df['Date'] = df['Date'].dt.date

        winHPLValues = 100 * df.Odd_H - 100
        winDPLValues = 100 * df.Odd_D - 100
        winAPLValues = 100 * df.Odd_A - 100
        losePLValues = -100

        df.loc[((df['Goals_H']) >  (df['Goals_A'])), 'Result_FT'] = 'H'
        df.loc[((df['Goals_H']) == (df['Goals_A'])), 'Result_FT'] = 'D'
        df.loc[((df['Goals_H'])  < (df['Goals_A'])), 'Result_FT'] = 'A'

        df['H'] = winHPLValues.where(df.Result_FT == 'H', other=losePLValues)
        df['D'] = winDPLValues.where(df.Result_FT == 'D', other=losePLValues)
        df['A'] = winAPLValues.where(df.Result_FT == 'A', other=losePLValues)

        no_of_days = 0

        matchDates = df.Date.unique()

        if no_of_days > 0:
            matchDates = (matchDates[-no_of_days:])

        df2 = pd.DataFrame()

        rowsIndex = []
        rowsDate = []
        rowsH = []
        rowsD = []
        rowsA = []

        count = 0
        for mDate in matchDates:
            count += 1
            rowsDate.append(mDate)
            rowsH.append(df.loc[df['Date'] == mDate]['H'].sum())
            rowsD.append(df.loc[df['Date'] == mDate]['D'].sum())
            rowsA.append(df.loc[df['Date'] == mDate]['A'].sum())

        df2['Date'] = rowsDate
        df2['H'] = rowsH
        df2['D'] = rowsD
        df2['A'] = rowsA

        df2 = df2.tail(101)
        df2 = df2.reset_index(drop=True)
        df2['Id'] = df2.reset_index()['index'].rename('index_copy')
        df2['Id'] = df2['Id'] + 1
        df2 = df2[['Id','Date','H','D','A']]

        df2['Hacu'] = df2.H.cumsum()
        df2['Dacu'] = df2.D.cumsum()
        df2['Aacu'] = df2.A.cumsum()

        df2['Hacu'].loc[0] = np.nan
        df2['Dacu'].loc[0] = np.nan
        df2['Aacu'].loc[0] = np.nan

        def weighted_mean_H(s):
            d = df2.loc[s.index, 'Hacu']
            w = df2.loc[s.index, 'Id']
            return (d * w).sum() / w.sum()

        def weighted_mean_H_C(s):
            d = df2.loc[s.index, 'waHC']
            w = df2.loc[s.index, 'Id']
            return (d * w).sum() / w.sum()

        def weighted_mean_D(s):
            d = df2.loc[s.index, 'Dacu']
            w = df2.loc[s.index, 'Id']
            return (d * w).sum() / w.sum()

        def weighted_mean_D_C(s):
            d = df2.loc[s.index, 'waDC']
            w = df2.loc[s.index, 'Id']
            return (d * w).sum() / w.sum()

        def weighted_mean_A(s):
            d = df2.loc[s.index, 'Aacu']
            w = df2.loc[s.index, 'Id']
            return (d * w).sum() / w.sum()

        def weighted_mean_A_C(s):
            d = df2.loc[s.index, 'waAC']
            w = df2.loc[s.index, 'Id']
            return (d * w).sum() / w.sum()

        df2['waH16'] = df2.rolling(16)['Hacu'].apply(weighted_mean_H, raw=False)
        df2['waH8'] = df2.rolling(8)['Hacu'].apply(weighted_mean_H, raw=False)
        df2['waHC'] = 2*df2.waH8-df2.waH16
        df2['waH4'] = df2.rolling(4)['waHC'].apply(weighted_mean_H_C, raw=False)

        df2['waD16'] = df2.rolling(16)['Dacu'].apply(weighted_mean_D, raw=False)
        df2['waD8'] = df2.rolling(8)['Dacu'].apply(weighted_mean_D, raw=False)
        df2['waDC'] = 2*df2.waD8-df2.waD16
        df2['waD4'] = df2.rolling(4)['waDC'].apply(weighted_mean_D_C, raw=False)

        df2['waA16'] = df2.rolling(16)['Aacu'].apply(weighted_mean_A, raw=False)
        df2['waA8'] = df2.rolling(8)['Aacu'].apply(weighted_mean_A, raw=False)
        df2['waAC'] = 2*df2.waA8-df2.waA16
        df2['waA4'] = df2.rolling(4)['waAC'].apply(weighted_mean_A_C, raw=False)

        df2['Hhull'] = df2['waH4']
        df2['Dhull'] = df2['waD4']
        df2['Ahull'] = df2['waA4']

        df2['Hdist'] = (df2.Hacu / df2.Hhull)
        df2['Ddist'] = (df2.Dacu / df2.Dhull)
        df2['Adist'] = (df2.Aacu / df2.Ahull)

        def r_function(s):
            f = s.loc[s.index.values[0]]
            l = s.loc[s.index.values[1]]
            return (l -f)/abs(f)

        df2['Hr'] = df2['Hhull'].rolling(2).apply(r_function, raw=False)
        df2['Dr'] = df2['Dhull'].rolling(2).apply(r_function, raw=False)
        df2['Ar'] = df2['Ahull'].rolling(2).apply(r_function, raw=False)

        def inc_H(s):
            x = df2.loc[s.index, 'Id']
            y = df2.loc[s.index, 'Hhull']
            slope, intercept = np.polyfit(x, y , 1)
            return slope

        def inc_D(s):
            x = df2.loc[s.index, 'Id']
            y = df2.loc[s.index, 'Dhull']
            slope, intercept = np.polyfit(x, y, 1)
            return slope

        def inc_A(s):
            x = df2.loc[s.index, 'Id']
            y = df2.loc[s.index, 'Ahull']
            slope, intercept = np.polyfit(x, y, 1)
            return slope

        df2['Hinc'] = df2['Hhull'].rolling(5).apply(inc_H, raw=False)
        df2['Dinc'] = df2['Dhull'].rolling(5).apply(inc_D, raw=False)
        df2['Ainc'] = df2['Ahull'].rolling(5).apply(inc_A, raw=False)

        df2['Hdp'] = df2['Hacu'].rolling(10).std()
        df2['Ddp'] = df2['Dacu'].rolling(10).std()
        df2['Adp'] = df2['Aacu'].rolling(10).std()

        df2['Hamp'] = df2['Hacu'].rolling(10).max() / df2['Hacu'].rolling(10).min()
        df2['Damp'] = df2['Dacu'].rolling(10).max() / df2['Dacu'].rolling(10).min()
        df2['Aamp'] = df2['Aacu'].rolling(10).max() / df2['Aacu'].rolling(10).min()

        df3 = pd.DataFrame()

        def normaliz(dfS):
            actual_value = (dfS.loc[dfS.index.values[4]])
            try:
                n = (actual_value - dfS.min()) / (dfS.max() - dfS.min())
                if math.isnan(n):
                    return 0
            except ZeroDivisionError:
                return 0
            return n

        df3['Id'] = df2['Id'].iloc[23:]

        df3['Hhull'] = df2['Hhull'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Dhull'] = df2['Dhull'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Ahull'] = df2['Ahull'].iloc[23:].rolling(5).apply(normaliz, raw=False)

        df3['Hdist'] = df2['Hdist'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Ddist'] = df2['Ddist'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Adist'] = df2['Adist'].iloc[23:].rolling(5).apply(normaliz, raw=False)

        df3['Hr'] = df2['Hr'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Dr'] = df2['Dr'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Ar'] = df2['Ar'].iloc[23:].rolling(5).apply(normaliz, raw=False)

        df3['Hinc'] = df2['Hinc'].rolling(5).apply(normaliz, raw=False)
        df3['Dinc'] = df2['Dinc'].rolling(5).apply(normaliz, raw=False)
        df3['Ainc'] = df2['Ainc'].rolling(5).apply(normaliz, raw=False)

        df3['Hdp'] = df2['Hdp'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Ddp'] = df2['Ddp'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Adp'] = df2['Adp'].iloc[23:].rolling(5).apply(normaliz, raw=False)

        df3['Hamp'] = df2['Hamp'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Damp'] = df2['Damp'].iloc[23:].rolling(5).apply(normaliz, raw=False)
        df3['Aamp'] = df2['Aamp'].iloc[23:].rolling(5).apply(normaliz, raw=False)

        df3['R'] = ''

        for index, row in df2.iterrows():
            if index > 26:
                try:
                    h = df2.iloc[(index+1)].H
                    d = df2.iloc[(index+1)].D
                    a = df2.iloc[(index+1)].A
                    if h > d and h > a:
                        df3.loc[index, 'R'] = 'H'
                    elif d > h and d > a:
                        df3.loc[index, 'R'] = 'D'
                    else:
                        df3.loc[index, 'R'] = 'A'
                except LookupError:
                    pass

        selected_date = df3.iloc[-1]

        distance_columns = ['Hhull', 'Dhull', 'Ahull', 'Hdist', 'Ddist', 'Adist', 'Hr', 'Dr', 'Ar', 'Hinc', 'Dinc', 'Ainc', 'Hdp',
                            'Ddp', 'Adp', 'Hamp', 'Damp', 'Aamp']

        def euclidean_distance(row):
            inner_value = 0
            for k in distance_columns:
                inner_value += (row[k] - selected_date[k]) ** 2
            return math.sqrt(inner_value)

        def euclidean_distance_tr(row):
            inner_value = 0
            for k in distance_columns:
                inner_value += abs(row[k] - selected_date[k])
            return inner_value

        df3['eucli'] = df3.apply(euclidean_distance_tr, axis=1)

        df4 = pd.DataFrame()
        df4['R'] = df3['R']
        df4['eucli'] = df3['eucli']
        df4.sort_values(by=['eucli'], inplace=True)

        teoria_dos_retornos = {}

        try:
            teoria_dos_retornos["League"] = liga
            teoria_dos_retornos["N1"] = df4[df4.eucli != 0]['R'].iloc[0]
        except:
            teoria_dos_retornos["N1"] = 0
        try:
            teoria_dos_retornos["N2"] = df4[df4.eucli != 0]['R'].iloc[1]
        except:
            teoria_dos_retornos["N2"] = 0
        try:
            teoria_dos_retornos["N3"] = df4[df4.eucli != 0]['R'].iloc[2]
        except:
            teoria_dos_retornos["N3"] = 0

        lista.append(teoria_dos_retornos)


In [ ]:
tt

In [ ]:
# Diferença absoluta entre as odds da equipe da casa e da visitante
df['VAR1'] = np.abs(df['Odd_H'] - df['Odd_A'])
# Medindo o ângulo em graus entre as odds da equipe da casa e da visitante
df['VAR2'] = np.degrees(np.arctan((df['Odd_A'] - df['Odd_H']) / 2))
# Medindo o ângulo em graus entre as odds do empate e da equipe visitante
df['VAR3'] = np.degrees(np.arctan((df['Odd_D'] - df['Odd_A']) / 2))
# Diferença absoluta entre as odds de mais de 2.5 gols e menos de 2.5 gols
df['VAR4'] = np.abs(df['Odd_Over25'] - df['Odd_Under25'])
# Ângulo em graus entre as odds de mais de 2.5 gols e menos de 2.5 gols
df['VAR5'] = np.degrees(np.arctan((df['Odd_Over25'] - df['Odd_Under25']) / 2))
# Diferença absoluta entre as odds de ambos os times marcarem (sim) e não marcarem (não)
df['VAR6'] = np.abs(df['Odd_BTTS_Yes'] - df['Odd_BTTS_No'])
# Ângulo em graus entre as odds de ambos os times marcarem (sim) e não marcarem (não)
df['VAR7'] = np.degrees(np.arctan((df['Odd_BTTS_Yes'] - df['Odd_BTTS_No']) / 2))
# Ângulo em graus entre a média das odds de vitória (casa e visitante) e as odds de mais de 2.5 gols
df['VAR8'] = np.degrees(np.arctan((df['Odd_Over25'] - ((df['Odd_H'] + df['Odd_A']) / 2)) / 2))
# Diferença absoluta entre a média das odds de menos de 2.5 gols e de ambos os times não marcarem vs. mais de 2.5 gols e ambos marcarem
df['VAR9'] = np.abs(((df['Odd_Under25'] + df['Odd_BTTS_No']) / 2) - ((df['Odd_Over25'] + df['Odd_BTTS_Yes']) / 2))
# Variáveis dfadas na média das odds
df['VAR10'] = (df['Odd_H'] + df['Odd_A'] + df['Odd_D']) / 3
df['VAR11'] = (df['Odd_Over25'] + df['Odd_Under25']) / 2
df['VAR12'] = (df['Odd_BTTS_Yes'] + df['Odd_BTTS_No']) / 2
# Diferença ponderada entre as odds de casa e de visitante, considerando o empate
df['VAR13'] = np.abs((df['Odd_H'] - df['Odd_A']) * df['Odd_D'])
# Soma das inversões das odds como medida de "certeza" do mercado
df['VAR14'] = (1/df['Odd_H']) + (1/df['Odd_A']) + (1/df['Odd_D'])
# Ângulo em graus entre a soma das odds de gol (mais de 2.5 e menos de 2.5) e as odds de BTTS
df['VAR15'] = np.degrees(np.arctan((df['VAR11'] - df['VAR12']) / 2))
# Relação entre as odds de mais/menos 2.5 gols e BTTS sim/não
df['VAR16'] = (df['Odd_Over25'] / df['Odd_Under25']) * (df['Odd_BTTS_Yes'] / df['Odd_BTTS_No'])
# Diferença entre a média das odds de vitória e a média das odds de gol/BTTS
df['VAR17'] = np.abs(df['VAR10'] - ((df['VAR11'] + df['VAR12']) / 2))
# Medida de disparidade entre as odds de vitória e as odds de gols
df['VAR18'] = np.sqrt((df['Odd_H'] - df['VAR11'])**2 + (df['Odd_A'] - df['VAR12'])**2)
# Ângulo em graus entre a diferença das odds de casa/visitante e a soma das odds de gol
df['VAR19'] = np.degrees(np.arctan(((df['Odd_H'] - df['Odd_A']) / (df['Odd_Over25'] + df['Odd_Under25'])) / 2))
# Razão entre as odds de mais de 2.5 gols e a média das odds de BTTS
df['VAR20'] = df['Odd_Over25'] / df['VAR12']
# Diferença absoluta entre as odds de vitória da equipe da casa e as odds de empate
df['VAR21'] = np.abs(df['Odd_H'] - df['Odd_D'])
# Diferença absoluta entre as odds de vitória da equipe visitante e as odds de empate
df['VAR22'] = np.abs(df['Odd_A'] - df['Odd_D'])
# Medida de tendência central para as odds de vitória (casa e visitante) e empate
df['VAR23'] = (df['Odd_H'] + df['Odd_A'] + df['Odd_D']) / 3
# Variabilidade nas odds de resultado (casa, empate, visitante)
df['VAR24'] = np.std([df['Odd_H'], df['Odd_A'], df['Odd_D']], axis=0)
# Coeficiente de variação para as odds de resultado, medindo a dispersão relativa
df['VAR25'] = df['VAR24'] / df['VAR23']
# Razão entre as odds de vitória da equipe da casa e as odds de vitória da equipe visitante
df['VAR26'] = df['Odd_H'] / df['Odd_A']
# Razão entre as odds de empate e as odds de mais de 2.5 gols
df['VAR27'] = df['Odd_D'] / df['Odd_Over25']

In [ ]:
# df['0x1'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 0 and row['Goals_A_FT'] == 1) else 0, axis=1)
# df['0x2'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 0 and row['Goals_A_FT'] == 2) else 0, axis=1) 
# df['0x3'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 0 and row['Goals_A_FT'] == 3) else 0, axis=1) 
# df['1x0'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 1 and row['Goals_A_FT'] == 0) else 0, axis=1)
# df['2x0'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 2 and row['Goals_A_FT'] == 0) else 0, axis=1) 
# df['3x0'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 3 and row['Goals_A_FT'] == 0) else 0, axis=1) 

# df['0x0'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 0 and row['Goals_A_FT'] == 0) else 0, axis=1)
# df['1x1'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 1 and row['Goals_A_FT'] == 1) else 0, axis=1)
# df['2x2'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 2 and row['Goals_A_FT'] == 2) else 0, axis=1)

# df['1x2'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 1 and row['Goals_A_FT'] == 2) else 0, axis=1)
# df['2x1'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] == 2 and row['Goals_A_FT'] == 1) else 0, axis=1)

# df['Goleada_H'] = df.apply(lambda row: 1 if (row['Goals_H_FT'] >= 4 and row['Goals_H_FT'] > row['Goals_A_FT']) else 0, axis=1) 
# df['Goleada_A'] = df.apply(lambda row: 1 if (row['Goals_A_FT'] >= 4 and row['Goals_A_FT'] > row['Goals_H_FT']) else 0, axis=1)

In [ ]:
# df.to_csv('Base_FutPythonPunter.csv',index=False)
df.to_excel('Base_FutPythonPunter.xlsx',index=False)